# Gemma Edu-Agent: Facilitando la Personalización a Escala

*Generación de andamiaje conceptual mediante modelos de frontera para docentes en entornos de alta densidad.*

## 1. El Problema: El Cuello de Botella Pedagógico
En América Latina, el sistema educativo público enfrenta aulas de más de 40 alumnos por docente. Explicar conceptos técnicos abstractos de física o matemáticas exige horas de re-explicaciones individualizadas. La personalización pedagógica a menudo resulta inviable por falta de tiempo. 

Asimismo, el uso no guiado de IA generativa presenta un reto: cuando los alumnos reciben resoluciones directas de sus tareas, el esfuerzo cognitivo disminuye. Nuestra hipótesis es que los docentes necesitan una herramienta que asista en la construcción de andamiaje mental, no un solucionador de ejercicios.

## 2. Solución Implementada: Analogías Estructuradas
**Gemma Edu-Agent** es un prototipo diseñado para ayudar al profesor. El MVP toma conceptos extraídos de literatura abierta (ej. *OpenStax Physics*) y asiste en la creación de analogías basadas en los intereses del estudiante (videojuegos, deportes, etc.).

El sistema no resuelve problemas numéricos; en su lugar, mapea explícitamente las reglas del concepto técnico a las reglas del dominio de interés del alumno, fomentando la comprensión relacional.

## 3. Arquitectura del Prototipo
La arquitectura actual demuestra el flujo de extremo a extremo utilizando los siguientes componentes:

1. **Recuperación Semántica Ligera (RAG):** El MVP utiliza `gemini-embedding-001` para generar embeddings de fragmentos de texto preseleccionados. La recuperación se realiza mediante un cálculo de similitud coseno en memoria utilizando `numpy`. Se usa el encoder de Gemini porque la familia Gemma no expone un endpoint de embeddings en Google AI Studio: EmbeddingGemma se distribuye para descarga propia o Vertex AI, no vía Gemini API. Todo el razonamiento pedagógico corre en Gemma 4. La recuperación es una similitud coseno en memoria con `numpy`, lo que priorizó reproducibilidad y un repositorio clonable sin dependencias nativas.
2. **Generación con Salidas Estructuradas (Structured Outputs):** Para controlar el formato de la respuesta, el agente se integra mediante la API generativa de Google forzando un esquema Pydantic. Esto restringe al modelo para emitir un documento JSON validable con campos específicos:
   * `technical_concept` y `source_citation`
   * `student_interest`
   * `conceptual_analogy`
   * `mapping_matrix` (Tabla relacional)
   * `verification_question` (Enfocada en transferencia, no en memoria)

## 4. Decisiones de Ingeniería para el MVP
* **Recuperación vs. Entrenamiento:** Se eligió un enfoque RAG en memoria en lugar de fine-tuning para reducir la latencia de implementación y asegurar que las respuestas se basaran explícitamente en el material de OpenStax proporcionado.
* **Reducción de Dependencias:** El reemplazo de bases de datos vectoriales complejas por una solución nativa basada en `numpy` permitió que el desarrollo y la ejecución del servidor Streamlit se mantuvieran estables en un entorno local (Windows) durante el límite de tiempo del evento.

## Enlaces del Proyecto
* **Demo en Vivo (Notebook Ejecutable):** [AÑADIR_URL]
* **Repositorio de Código Público:** https://github.com/ricardomartinezsau-jpg/hackday
* **Video de Demostración:** [AÑADIR_URL_YOUTUBE]

## Honestidad técnica sobre el uso de Gemma 4

**Gemma 4 es el motor de razonamiento pedagógico del producto.** Comprender el
concepto físico, elegir el anclaje analogógico, construir el mapeo y formular la
pregunta de verificación ocurren íntegramente en `gemma-4-31b-it`.

Dos precisiones que preferimos declarar antes de que las pregunte un juez:

- Usamos **structured output** (`responseSchema`), no *function calling*. Son
  capacidades distintas y solo la primera está en juego en este prototipo.
- **No usamos "la familia completa de Gemma".** El embedding es Gemini, porque
  Gemma no ofrece uno por API.

## Lo que se rompió y cómo se arregló

Tres hallazgos empíricos contra el endpoint real, ninguno documentado de antemano:

1. **`system_instruction` + `responseSchema` cuelga la petición.** Movimos el rol
   al prompt, que es además la forma en que Gemma fue entrenado para recibirlo.
2. **Los esquemas con objetos anidados disparan bucles de repetición.** Un
   `ARRAY` de objetos hacía degenerar la generación hasta agotar el presupuesto
   de tokens y truncar el JSON. Aplanar `mapping_matrix` a cadenas
   `"academico :: analogico"` bajó la latencia de ~120 s a ~9 s y llevó la tasa
   de éxito de 1/3 a 3/3.
3. **La decodificación restringida no elimina la degeneración estocástica.** Un
   techo ceñido de `maxOutputTokens` hace que el fallo sea barato, y tres
   reintentos a temperatura descendente lo vuelven invisible en la demo.


# Gemma Edu-Agent — demo ejecutable

Corre el pipeline completo sin levantar Streamlit. Funciona en Google Colab o
Kaggle Notebooks. Solo necesitas una API key de Google AI Studio.

In [ ]:
!pip install requests pydantic numpy -q

In [ ]:
import json
import numpy as np
import requests
from typing import List
from pydantic import BaseModel, Field, ValidationError

# En Kaggle: Add-ons > Secrets > nuevo secreto llamado GEMINI_API_KEY.
# En Colab o local: define la variable de entorno GEMINI_API_KEY.
# Nunca escribas la clave directamente: este cuaderno es publico.
import os
try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY", "TU_API_KEY_AQUI")

API_BASE = "https://generativelanguage.googleapis.com/v1beta/models/"
MODEL = "gemma-4-31b-it"        # motor de razonamiento pedagogico
EMBED = "gemini-embedding-001"  # Gemma no expone endpoint de embeddings

## 1. Recuperación semántica sobre el corpus OER

El corpus son secciones de OpenStax Physics 2e (CC BY 4.0). Los vectores se
comparan con similitud coseno en memoria: para una decena de secciones el
producto punto es exacto y evita dependencias nativas.

In [ ]:
CORPUS = [
    "## 8.1 Momento Lineal. El momento lineal p de un objeto es el producto de su masa "
    "por su velocidad: p = m*v. Es una magnitud vectorial.",
    "## 8.2 Conservacion del Momento Lineal. Si la fuerza neta externa sobre un sistema "
    "de particulas es cero, el momento lineal total permanece constante: "
    "m1*v1_i + m2*v2_i = m1*v1_f + m2*v2_f.",
    "## 8.3 Colisiones Elasticas e Inelasticas. En una colision elastica se conservan el "
    "momento y la energia cinetica. En una inelastica solo se conserva el momento.",
]

def embed(text, task_type):
    r = requests.post(
        API_BASE + EMBED + ":embedContent?key=" + API_KEY,
        json={"model": "models/" + EMBED,
              "content": {"parts": [{"text": text}]},
              "taskType": task_type},
        timeout=60)
    r.raise_for_status()
    return np.array(r.json()["embedding"]["values"], dtype=np.float32)

DOC_VECTORS = [embed(d, "RETRIEVAL_DOCUMENT") for d in CORPUS]

def retrieve(query):
    q = embed(query, "RETRIEVAL_QUERY")
    sims = [float(np.dot(q, d) / (np.linalg.norm(q) * np.linalg.norm(d))) for d in DOC_VECTORS]
    return CORPUS[int(np.argmax(sims))]

print("Indice listo:", len(DOC_VECTORS), "secciones")

## 2. Contrato de salida

Pydantic valida lo que devuelve Gemma. El `mapping_matrix` es una lista **plana**
de cadenas `"academico :: analogico"`: anidar objetos dentro del esquema
provocaba bucles de repetición que truncaban el JSON.

In [ ]:
class MappingObject(BaseModel):
    academico: str
    analogico: str

class PedagogicalAnalogy(BaseModel):
    technical_concept: str
    source_citation: str
    student_interest: str
    conceptual_analogy: str
    mapping_matrix: List[str] = Field(description="Formato 'academico :: analogico'")
    verification_question: str

    def mapping_rows(self):
        rows = []
        for item in self.mapping_matrix:
            left, _, right = item.partition("::")
            if right.strip():
                rows.append(MappingObject(academico=left.strip(), analogico=right.strip()))
        return rows

RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "technical_concept": {"type": "STRING"},
        "source_citation": {"type": "STRING"},
        "student_interest": {"type": "STRING"},
        "conceptual_analogy": {"type": "STRING"},
        "mapping_matrix": {"type": "ARRAY", "items": {"type": "STRING"}},
        "verification_question": {"type": "STRING"},
    },
    "required": ["technical_concept", "source_citation", "student_interest",
                 "conceptual_analogy", "mapping_matrix", "verification_question"],
}

## 3. Generación con Gemma 4

El rol va dentro del prompt, no en `system_instruction`: esa combinación cuelga
la petición contra el endpoint de Gemma. Tres reintentos a temperatura
descendente absorben la degeneración estocástica.

In [ ]:
ROLE = ("Eres un experto en pedagogia constructivista. Explicas conceptos tecnicos "
        "mediante analogias ancladas en el mundo real del estudiante. Nunca resuelves "
        "el problema por el alumno: construyes el andamiaje para que lo resuelva solo. "
        "Basas tu respuesta UNICAMENTE en el texto fuente proporcionado.")

def generar_analogia(contexto, interes):
    prompt = ROLE + """

Contexto Recuperado del Libro de Texto (OER):
""" + contexto + """

Interes del Estudiante:
""" + interes + """

Genera una analogia pedagogica estructurada respetando estos limites:
- conceptual_analogy: maximo 90 palabras, en segunda persona. No repitas frases.
- mapping_matrix: exactamente 4 cadenas con el formato "academico :: analogico".
- source_citation: una frase literal del contexto anterior mas su numero de seccion.
- verification_question: una sola pregunta, sin respuesta."""

    ultimo = None
    for temp in (0.7, 0.4, 0.2):
        try:
            r = requests.post(
                API_BASE + MODEL + ":generateContent?key=" + API_KEY,
                json={"contents": [{"parts": [{"text": prompt}]}],
                      "generationConfig": {"responseMimeType": "application/json",
                                           "responseSchema": RESPONSE_SCHEMA,
                                           "temperature": temp, "topP": 0.95,
                                           "maxOutputTokens": 900}},
                timeout=180)
            r.raise_for_status()
            texto = r.json()["candidates"][0]["content"]["parts"][0]["text"]
            return PedagogicalAnalogy.model_validate_json(texto)
        except (ValidationError, requests.RequestException, KeyError, IndexError) as e:
            ultimo = e
    raise RuntimeError("Gemma 4 no produjo una analogia valida: %s" % ultimo)

## 4. Pipeline completo

Cambia `interes` por cualquier cosa —fútbol, cocina, K-pop, mecánica— y observa
cómo se reancla el mismo concepto físico.

In [ ]:
tema = "Conservacion del momento lineal"
interes = "produccion musical y beats de hip hop"

contexto = retrieve(tema)
print("FUENTE RECUPERADA (no generada):")
print(" ", contexto[:150], "...")
print()

r = generar_analogia(contexto, interes)

print("CONCEPTO:", r.technical_concept)
print("CITA:    ", r.source_citation)
print()
print("ANALOGIA:")
print(" ", r.conceptual_analogy)
print()
print("MATRIZ DE MAPEO:")
for fila in r.mapping_rows():
    print("  %-38s -> %s" % (fila.academico, fila.analogico))
print()
print("PREGUNTA DE VERIFICACION:")
print(" ", r.verification_question)